# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NameRectified/flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Lane:** CTR / Engagement Opportunity Scoring (combined)

I am choosing this lane because I think flyrank's core decision revolves around prioritizing the pages that a human should review first from a pool of thousands of pages. Also a page can rank well under - captured clicks due to reasons like intent mismatch but may fail on page i.e weak engagement. So this lane lets me explore both the CTR and engagement and hence I am choosing it.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. The question: decision, action, cost of a wrong call

- Decision: It helps decide which high exposure pages a content reviewer should open first. for a CTR fix (title, meta, snippet, intent match) or an engagement fix (on-page structure, depth, intent fulfillment)?

- Who takes the action: A content reviewer or editor takes the required action.

- Cost of a wrong call:
    - False priority (noise): Review spends time on pages with low volume or low CTR
    - Missed opportunity: A page that could have improved loses the opportunity to be improved.
- Why ML: With rules we can get a good baseline but not high performance. It is not possible to represent the multiple signals like interaction of the content type, keyword demand, position etc using conditional rules.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Quick look at the data (2-3 real numbers)
The code cell below prints the counts; three headline numbers:

1. 3,140 pages (10.5%) have enough search volume (`impressions_90d ≥ 500`) and sit 0.1 percentage points below their position-tier median CTR — real CTR-review candidates, not low-impression noise.
2. ~4,919 pages (16.4%) have `sessions_90d ≥ 50` but `engagement_rate < 30%` — people arrive, but on-page engagement is weak.
3. ~427 pages (1.4%) hit both signals — highest-priority "fix snippet *and* on-page" cases for the top of the queue.


In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from pathlib import Path

import pandas as pd

DATA_PATH = Path("../../data/raw/content_refresh_anonymized.csv")
df = pd.read_csv(DATA_PATH)


ctr_base = df[(df["avg_position"] > 0) & (df["impressions_90d"] >= 500)].copy()
ctr_base["tier_med_ctr"] = ctr_base.groupby("position_tier")["ctr"].transform("median")
ctr_base["ctr_gap_pp"] = ctr_base["tier_med_ctr"] - ctr_base["ctr"]
ctr_opp = ctr_base[ctr_base["ctr_gap_pp"] > 0.1]


eng_opp = df[(df["sessions_90d"] >= 50) & (df["engagement_rate"] < 30)]

both = ctr_opp.merge(eng_opp[["content_id"]], on="content_id")

print(f"Inventory: {len(df):,} pages across {df['client_id'].nunique()} clients")
print(
    f"CTR gap candidates (impr>=500, >0.1pp below tier): "
    f"{len(ctr_opp):,} ({100 * len(ctr_opp) / len(df):.1f}%)"
)
print(
    f"Engagement gap candidates (sessions>=50, engagement<30%): "
    f"{len(eng_opp):,} ({100 * len(eng_opp) / len(df):.1f}%)"
)
print(f"Both signals on same page: {len(both):,} ({100 * len(both) / len(df):.1f}%)")
print(f"Median CTR gap (pp) among CTR candidates: {ctr_opp['ctr_gap_pp'].median():.2f}")
print(f"Median sessions among engagement candidates: {eng_opp['sessions_90d'].median():.0f}")


Inventory: 30,000 pages across 32 clients
CTR gap candidates (impr>=500, >0.1pp below tier): 3,140 (10.5%)
Engagement gap candidates (sessions>=50, engagement<30%): 4,919 (16.4%)
Both signals on same page: 427 (1.4%)
Median CTR gap (pp) among CTR candidates: 0.17
Median sessions among engagement candidates: 108


## 4. Careful words: what I can and can't claim

What I can claim:
- Observed: Pages with high impressions but CTR below their position-tier peers, and pages with meaningful sessions but low engagement rate, exist at scale in this inventory
- Decision-support: A combined score can rank a review queue with reason codes (ctr_gap, engagement_gap, both) so a human spends limited time on the highest-exposure candidates first and a learned score may beat a plain tier median rule on precision@K.

What I can't claim:
- Editing a title or page caused CTR or engagement to recover
- Predicting google or reverse engineering ranking algorithms

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.